# 1. Business Problem? 

Business problem → analysis → recommendation

* The e-commerce platform lacks visibility into user conversion behavior across the purchase funnel.
* This project aims to construct a user-level funnel, quantify drop-offs, and uncover behavioral patterns that impact conversion.

# 2. Domain?

E-commerce / Product Analytics

* This project falls under E-commerce Product Analytics, focusing on user behavior analysis across the online purchase funnel (view → add-to-cart → transaction).
* The objective is to improve conversion rates and optimize the user journey.

# 3. Data Understanding 

* Load dataset (Python)
* Clean + understand events
* Build session logic
* Create funnel table (SQL or Pandas)

* The behaviour data, i.e. events like clicks, add to carts, transactions, represent interactions that were collected over a period of 4.5 months.
* There are three files:
  * event.csv - “view”, “addtocart” or “transaction”.
  * item_properties.csv - corresponding properties of the events.
  * category_tree.csv - file specifies a child categoryId and the corresponding parent.

In [ ]:
import os
print(os.getcwd())

Step 1 — Load the dataset

In [ ]:
import pandas as pd

# Load events dataset
df = pd.read_csv("../Dataset/Raw dataset/events.csv")

# View data
print(df.head())

# Check shape
print(df.shape)

Step 2 — Understand columns

| Column        | Meaning                       |
| ------------- | ----------------------------- |
| timestamp     | Unix timestamp (milliseconds) |
| visitorid     | User ID                       |
| event         | view/addtocart/transaction    |
| itemid        | Product ID                    |
| transactionid | Purchase ID                   |


In [ ]:
df.event.value_counts()

In [ ]:
df[df.event=='transaction'].head()

In [ ]:
df.describe(include='all')

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
print(df[df.event=='view'].shape[0] + df[df.event=='addtocart'].shape[0])
print(df.transactionid.isnull().sum())

In [ ]:
print(df.shape)
print(df.visitorid.nunique())

In [ ]:
print(df.shape)
print(df.itemid.nunique())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go


In [ ]:
# Raw event count distribution
import matplotlib.pyplot as plt
import seaborn as sns

order = ['view', 'addtocart', 'transaction']
colors = ['#4C78A8', '#F58518', '#54A24B']

event_counts = (
    df['event']
    .value_counts()
    .reindex(order)
    .reset_index()
)
event_counts.columns = ['event', 'count']

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=event_counts,
    x='event',
    y='count',
    hue='event',
    order=order,
    hue_order=order,
    palette=colors,
    legend=False,
    ax=ax
)

ax.set_title('Raw Event Count Distribution', fontsize=13, pad=12)
ax.set_xlabel('Event type')
ax.set_ylabel('Number of events')
ax.ticklabel_format(style='plain', axis='y')

# Add count labels manually for better compatibility across matplotlib/seaborn versions.
for patch, count in zip(ax.patches, event_counts['count']):
    x = patch.get_x() + patch.get_width() / 2
    y = patch.get_height()
    ax.text(
        x,
        y + (event_counts['count'].max() * 0.015),
        f'{count:,.0f}',
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

ax.set_ylim(0, event_counts['count'].max() * 1.12)
sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
flow_count = [
    df[df['event'] == 'view'].shape[0],
    df[df['event'] == 'addtocart'].shape[0],
    df[df['event'] == 'transaction'].shape[0]
]

In [ ]:
stages = ['view product', 'add to cart', 'purchased product']
fig = go.Figure(go.Funnel( x=flow_count, y=stages, textinfo = 'value+percent initial'))
fig.show()

### Raw event chart interpretation

These charts show raw event volume before sessionization.

Key observations:

- `view` events dominate the dataset, which indicates that most recorded user activity is browsing behavior.
- `addtocart` events are much smaller in volume compared with `view` events.
- `transaction` events are the smallest event category.
- This gives an early signal that the funnel narrows sharply after product views.

Important caveat:

These charts are based on event counts, not unique visitors or sessions. One visitor can generate multiple view events, and raw event counts do not prove that the same visitor moved through `view -> addtocart -> transaction` in one journey.

Therefore, these charts should be interpreted as an initial event-level overview, not as true funnel conversion. True conversion and drop-off will be measured after sessionization, when the data is transformed into a session-level funnel table.


# 4. Data Preparation

4.1 Data Cleaning
* Remove duplicates 
* Handle missing values 
* Convert timestamps
  
4.2 Sessionization (CRITICAL)
* Define session (30-min inactivity rule) 
* Create session_id
  
4.3 Funnel Table Creation
* At session level

| user | session | view | cart | purchase |

Step 3 — Remove duplicate event rows if present

In [ ]:
# Check and remove exact duplicate rows from the event-level dataframe
# This is done before sessionization so each session is built from clean event records.
event_duplicate_rows_before_cleanup = df.duplicated().sum()
print(f"Duplicate event rows before cleanup: {event_duplicate_rows_before_cleanup:,}")

if event_duplicate_rows_before_cleanup > 0:
    df = df.drop_duplicates().copy()
    print(f"Duplicate event rows removed: {event_duplicate_rows_before_cleanup:,}")
else:
    print("No duplicate event rows found.")

print(f"Event dataframe shape after duplicate cleanup: {df.shape}")


Step 4 — Convert timestamp

In [ ]:
# Convert milliseconds to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

# Verify
print(df['timestamp'].head())

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
df[df.visitorid==257597].head()

In [ ]:
print(df.timestamp.min(),df.timestamp.max())

## Why convert event-level data into a session-level funnel table?

The raw events dataset is at the event level, where each row represents one user action such as a product view, add-to-cart, or transaction. This is useful for understanding individual actions, but raw event counts alone do not clearly show how users move through the funnel.

For funnel analysis, we need a clear unit of analysis. In this project, the unit is a session, which represents one user visit or browsing journey. Creating a `session_key` groups all events from the same visitor into the same session based on the 30-minute inactivity rule.

After generating session keys, the event-level table can be transformed into a session-level funnel table where each row represents one session and the funnel columns show whether that session included a view, add-to-cart, or purchase.

This conversion helps answer business questions such as:

- How many sessions included a product view?
- How many sessions progressed to add-to-cart?
- How many sessions ended in purchase?
- Where are users dropping off in the funnel?
- Which sessions are browse-only, cart-abandoned, or completed purchase sessions?

In short, sessionization turns many individual event rows into visit-level journeys, and the funnel table makes conversion and drop-off analysis easier to measure and interpret.


# Sessionization

Step 5 — Sort correctly

Sessionization depends on chronological order.

In [ ]:
df = df.sort_values(
    by=['visitorid', 'timestamp']
)

In [ ]:
df.head()

In [ ]:
df[df.visitorid==1].head()

Step 6 — Calculate time difference between events

In [ ]:
df[df.visitorid == 0].groupby('visitorid')['timestamp'].diff().head()

In [ ]:
# Calculate time difference per user
df['time_diff'] = (
    df.groupby('visitorid')['timestamp']
      .diff()
)

In [ ]:
df.head()

Step 7 — Create new session flag

30 minutes = 1800 seconds

In [ ]:
# Create session break flag
df['new_session'] = (
    df['time_diff'].isna()
    |
    (df['time_diff'].dt.total_seconds() > 1800)
)

In [ ]:
df.head()

In [ ]:
df[df.new_session==True].head()

In [ ]:
df.new_session.value_counts()

In [ ]:
df.new_session.isnull().sum()

Step 8 — Generate session IDs

* “Each True in new_session indicates the start of a new session. Using cumulative sum within each user creates sequential session numbering efficiently without loops.”

In [ ]:
# Create cumulative session count
df['session_id'] = (
    df.groupby('visitorid')['new_session']
      .cumsum()
)

In [ ]:
df['session_id'].unique()

In [ ]:
df['session_id'].value_counts()

Step 9 — Create unique session key

In [ ]:
df['session_key'] = (
    df['visitorid'].astype(str)
    + "_"
    + df['session_id'].astype(str)
)

In [ ]:
df.head()

In [ ]:
df.session_key.nunique()

In [ ]:
df[['visitorid','event','timestamp','session_key']].head(15)

# Session Key validation check

Check that each session key belongs to only one user

- Two different users should never share the same `session_key`.
- A result of `1` confirms that each session belongs to only one visitor.


In [ ]:
df.groupby('session_key')['visitorid'].nunique().max()

In [ ]:
df.head()

In [ ]:
df[['visitorid','event','timestamp','session_key']].head(3)

* Each visitor can have same or multiple session keys but two different users never share same session_key

In [ ]:
df.visitorid.nunique()

In [ ]:
df.session_key.nunique()

In [ ]:
df.isnull().sum()

# Funnel Table

Step 1 — Create event flags

In [ ]:
# Create binary flags for events
df['viewed'] = (df['event'] == 'view').astype(int)

df['carted'] = (df['event'] == 'addtocart').astype(int)

df['purchased'] = (df['event'] == 'transaction').astype(int)

In [ ]:
df.head()

In [ ]:
df[df.event=='transaction'].head()

In [ ]:
df[df.visitorid==0].groupby('session_key').agg({
          'visitorid': 'first',
          'viewed': 'max',
          'carted': 'max',
          'purchased': 'max',
          'timestamp': ['min', 'max']
      })

Step 2 — Aggregate at session level

In [ ]:
funnel_df = (
    df.groupby('session_key')
      .agg({
          'visitorid': 'first',
          'viewed': 'max',
          'carted': 'max',
          'purchased': 'max',
          'timestamp': ['min', 'max']
      })
)

In [ ]:
funnel_df.head()

Step 3 — Flatten column names

In [ ]:
# Flatten columns
funnel_df.columns = [
    'visitorid',
    'viewed',
    'carted',
    'purchased',
    'session_start',
    'session_end'
]

funnel_df = funnel_df.reset_index()
funnel_df.head()

Step 4 — Create session duration

In [ ]:
funnel_df['session_duration_mins'] = (
    (
        funnel_df['session_end']
        - funnel_df['session_start']
    ).dt.total_seconds() / 60
)

In [ ]:
funnel_df.head()

In [ ]:
funnel_df[['viewed', 'carted', 'purchased']].mean()

In [ ]:
# View data
print(funnel_df.head())

# Check shape
print(funnel_df.shape)

In [ ]:
# Check info
print(funnel_df.info())
# Check dtypes
print(funnel_df.dtypes)

# Funnel Table Validation Checks

Before moving to EDA, validate whether the session-level funnel table behaves as expected. These checks help identify duplicate records, missing funnel stages, and sessions where user behavior does not follow a simple view-to-cart-to-purchase path.

In [ ]:
funnel_df.head()

In [ ]:
# Unique session key
print(funnel_df.shape)
print(funnel_df.session_key.nunique())

In [ ]:
funnel_df.duplicated().sum()

In [ ]:
# Check for sessions where cart or purchase happened without a view in the same session
cart_without_view = funnel_df[
    (funnel_df['carted'] == 1)
    & (funnel_df['viewed'] == 0)
]

purchase_without_view = funnel_df[
    (funnel_df['purchased'] == 1)
    & (funnel_df['viewed'] == 0)
]

purchase_without_cart = funnel_df[
    (funnel_df['purchased'] == 1)
    & (funnel_df['carted'] == 0)
]

print(f"Cart without view sessions: {cart_without_view.shape[0]:,}")
print(f"Purchase without view sessions: {purchase_without_view.shape[0]:,}")
print(f"Purchase without cart sessions: {purchase_without_cart.shape[0]:,}")


In [ ]:
# Summarize validation checks as a small table
validation_summary = pd.DataFrame({
    'check': [
        'Cart without view in same session',
        'Purchase without view in same session',
        'Purchase without cart in same session'
    ],
    'count': [
        cart_without_view.shape[0],
        purchase_without_view.shape[0],
        purchase_without_cart.shape[0]
    ]
})

validation_summary


If these counts are greater than zero, it does not automatically mean the data is wrong. It may reflect cross-session behavior, missing tracking events, direct checkout flows, or the limits of using a 30-minute session window. We should note these cases before interpreting funnel conversion rates.

Next, check all possible funnel-stage combinations at the session level. This shows how many sessions are clean full-funnel sessions versus partial or unusual paths.

* view only - 1 0 0(expected)
* view + cart - 1 1 0(expected)
* view + purchase - 1 0 1(drill down - may be already added in the cart)
* view + cart + purchase - 1 1 1 (expected perfect!)
* cart only - 0 1 0(drill down - viewed in the previous session)
* cart + purchase - 0 1 1(drill down - viewed in the previous session)
* purchase only - 0 0 1(drill down - viewed and added to cart in the previous session)

In [ ]:
# Count all session-level funnel stage combinations
funnel_stage_combinations = (
    funnel_df
    .groupby(['viewed', 'carted', 'purchased'])
    .size()
    .reset_index(name='session_count')
    .sort_values('session_count', ascending=False)
)

funnel_stage_combinations


### Funnel-stage combination interpretation

The funnel-stage combination counts are internally consistent with the earlier validation checks:

- `cart without view` = `cart only` + `cart + purchase without view` = `4,409 + 789 = 5,198`
- `purchase without view` = `cart + purchase without view` + `purchase only` = `789 + 696 = 1,485`
- `purchase without cart` = `view + purchase without cart` + `purchase only` = `1,669 + 696 = 2,365`

Key observations:

- Most sessions are `view only`, which is expected for ecommerce browsing behavior.
- `view + cart` sessions without purchase represent cart abandonment opportunities.
- `view + cart + purchase` sessions are clean full-funnel sessions.
- Sessions with cart or purchase but no view may reflect returning users, prior-session browsing, direct cart/checkout access, or missing view tracking.
- Sessions with purchase but no cart may reflect direct checkout, cart activity in a previous session, missing add-to-cart tracking, or the effect of the 30-minute session rule.

Conclusion: the session-level funnel table is valid for EDA, but it should be described carefully. This is a session-level behavioral funnel, not a guaranteed ordered event-sequence funnel.


Finally, validate session duration. Since the data was sorted before sessionization, session duration should never be negative.

In [ ]:
# Check for invalid session durations
negative_session_durations = (funnel_df['session_duration_mins'] < 0).sum()
print(f"Sessions with negative duration: {negative_session_durations:,}")

funnel_df['session_duration_mins'].describe()


In [ ]:
# Explore sessions longer than 30 minutes
# These are valid if each consecutive event gap stayed within the 30-minute inactivity threshold.
session_durations_greater_than_30 = (funnel_df['session_duration_mins'] > 30).sum()
print(f"Sessions with duration more than 30 mins: {session_durations_greater_than_30:,}")


This is valid:
* Rule : Start a new session only when the gap between two consecutive events from the same user is more than 30 minutes.
* That means a session can last longer than 30 minutes if the user keeps interacting within 30-minute gaps.

### Session duration validation note

A session duration greater than 30 minutes is not automatically invalid.

The 30-minute session rule means a new session starts only when the gap between two consecutive events from the same visitor is more than 30 minutes. It does not mean the total session duration must be 30 minutes or less.

Example:

| Event time | Event |
| --- | --- |
| 10:00 | view |
| 10:20 | view |
| 10:45 | addtocart |
| 11:10 | transaction |

Each event gap is within 30 minutes:

- 10:00 to 10:20 = 20 minutes
- 10:20 to 10:45 = 25 minutes
- 10:45 to 11:10 = 25 minutes

So all events belong to the same session, even though the total session duration is 70 minutes.

For validation, negative session duration would be invalid. A stronger sessionization check would be to confirm that no internal event gap within a session is greater than 30 minutes.


# Optional Deep-Dive: Cross-Session Prior-Step Validation

Some sessions contain `addtocart` without a `view`, or `transaction` without an `addtocart`. These are not automatically errors. The missing prior step may have happened in an earlier session for the same visitor and product.

The next checks validate two cases:

1. For sessions where a product was added to cart without a view, check whether the same visitor viewed the same product in a previous session.
2. For sessions where a product was purchased without add-to-cart, check whether the same visitor added the same product to cart in a previous session.


In [ ]:
def summarize_prior_event_in_previous_session(target_events, prior_event_name, prior_step_label):
    # Faster targeted lookup:
    # 1. Keep only visitor-product pairs from the target anomaly events.
    # 2. Pull only prior-event rows for those same visitor-product pairs.
    # 3. Check whether a prior event exists in an earlier session_id.
    target_events = target_events.reset_index(drop=False).rename(columns={'index': 'target_event_index'})

    if target_events.empty:
        return pd.DataFrame({
            'metric': [
                'Target events',
                f'Had same visitor-product {prior_step_label} in previous session',
                f'No same visitor-product {prior_step_label} found in previous session'
            ],
            'event_count': [0, 0, 0],
            'share_of_target_events': ['0.00%', '0.00%', '0.00%']
        })

    target_pairs = target_events[['visitorid', 'itemid']].drop_duplicates()

    prior_events = (
        df[df['event'] == prior_event_name][['visitorid', 'itemid', 'session_id']]
        .drop_duplicates()
        .merge(target_pairs, on=['visitorid', 'itemid'], how='inner')
    )

    prior_matches = target_events[
        ['target_event_index', 'visitorid', 'itemid', 'session_id']
    ].merge(
        prior_events,
        on=['visitorid', 'itemid'],
        how='left',
        suffixes=('_target', '_prior')
    )

    matched_target_events = prior_matches.loc[
        prior_matches['session_id_prior'] < prior_matches['session_id_target'],
        'target_event_index'
    ].drop_duplicates()

    matched_count = matched_target_events.shape[0]
    total_count = target_events.shape[0]
    unmatched_count = total_count - matched_count

    summary = pd.DataFrame({
        'metric': [
            'Target events',
            f'Had same visitor-product {prior_step_label} in previous session',
            f'No same visitor-product {prior_step_label} found in previous session'
        ],
        'event_count': [total_count, matched_count, unmatched_count]
    })

    summary['share_of_target_events'] = (
        summary['event_count'] / total_count
    ).map(lambda x: f'{x:.2%}')

    return summary


## 1. Add-to-cart without view: did the visitor view the same product in a previous session?

In [ ]:
# Target add-to-cart events from sessions where cart happened without a view in the same session
cart_without_view_sessions = funnel_df[
    (funnel_df['carted'] == 1)
    & (funnel_df['viewed'] == 0)
]['session_key']

cart_without_view_events = df[
    (df['session_key'].isin(cart_without_view_sessions))
    & (df['event'] == 'addtocart')
].copy()

cart_prior_view_summary = summarize_prior_event_in_previous_session(
    target_events=cart_without_view_events,
    prior_event_name='view',
    prior_step_label='view'
)

cart_prior_view_summary


## 2. Purchase without add-to-cart: did the visitor add the same product to cart in a previous session?

In [ ]:
# Target purchase events from sessions where purchase happened without add-to-cart in the same session
purchase_without_cart_sessions = funnel_df[
    (funnel_df['purchased'] == 1)
    & (funnel_df['carted'] == 0)
]['session_key']

purchase_without_cart_events = df[
    (df['session_key'].isin(purchase_without_cart_sessions))
    & (df['event'] == 'transaction')
].copy()

purchase_prior_cart_summary = summarize_prior_event_in_previous_session(
    target_events=purchase_without_cart_events,
    prior_event_name='addtocart',
    prior_step_label='cart'
)

purchase_prior_cart_summary


Interpretation guide:

- If many add-to-cart events without same-session views had a prior same-product view, then those sessions likely represent returning users or cross-session product consideration.
- If many purchase events without same-session carts had a prior same-product add-to-cart, then those purchases likely represent users returning to complete checkout from a previous session.
- If prior same-product events are not found, the behavior may reflect missing tracking, direct cart or checkout access, alternate purchase flows, or products viewed/carted outside the available observation window.


# Optional Deep-Dive: Event Order Validation For Full-Funnel Sessions

The session-level funnel table confirms whether a session contains `view`, `addtocart`, and `transaction`, but it does not automatically prove those events happened in the expected order.

Before moving to EDA, validate clean full-funnel sessions to check whether the first event timestamps follow this sequence:

`view -> addtocart -> transaction`


In [ ]:
# Validate event order inside clean full-funnel sessions
full_funnel_sessions = funnel_df[
    (funnel_df['viewed'] == 1)
    & (funnel_df['carted'] == 1)
    & (funnel_df['purchased'] == 1)
]['session_key']

full_funnel_event_times = (
    df[
        (df['session_key'].isin(full_funnel_sessions))
        & (df['event'].isin(['view', 'addtocart', 'transaction']))
    ]
    .groupby(['session_key', 'event'])['timestamp']
    .min()
    .unstack()
    .reset_index()
)

full_funnel_event_times['valid_event_order'] = (
    (full_funnel_event_times['view'] <= full_funnel_event_times['addtocart'])
    & (full_funnel_event_times['addtocart'] <= full_funnel_event_times['transaction'])
)

valid_order_count = full_funnel_event_times['valid_event_order'].sum()
invalid_order_count = (~full_funnel_event_times['valid_event_order']).sum()

event_order_summary = pd.DataFrame({
    'metric': [
        'Full-funnel sessions',
        'Valid order: view -> cart -> purchase',
        'Invalid or unexpected order'
    ],
    'session_count': [
        full_funnel_event_times.shape[0],
        valid_order_count,
        invalid_order_count
    ]
})

event_order_summary['share_of_full_funnel_sessions'] = (
    event_order_summary['session_count'] / full_funnel_event_times.shape[0]
).map(lambda x: f'{x:.2%}')

event_order_summary


In [ ]:
funnel_df[
    (funnel_df['viewed'] == 1)
    & (funnel_df['carted'] == 1)
    & (funnel_df['purchased'] == 1)
]['session_key']

### Event-order validation interpretation

Among full-funnel sessions, 93.43% follow the expected `view -> addtocart -> transaction` event order.

This supports using the strict funnel for directional analysis because most sessions that contain all three funnel steps follow the expected sequence.

The remaining 6.57% of full-funnel sessions have invalid or unexpected ordering. These sessions should be treated as interpretation limitations rather than removed automatically.

Possible reasons include:

- users performing multiple views, carts, or purchases in the same session
- tracking delays or logging order issues
- users purchasing one item and viewing or carting another item in the same session
- mixed-product behavior within the same session

Important caveat: this validation checks event order at the session level, not at the same-product level. Because a session can contain multiple products, some unexpected ordering may be explained by users interacting with different items during the same session.


In [ ]:
# Inspect a few sessions where the expected event order is not followed
full_funnel_event_times[
    full_funnel_event_times['valid_event_order'] == False
].head(10)


Interpretation guide:

- If most full-funnel sessions follow `view -> addtocart -> transaction`, the strict funnel interpretation is reliable for those sessions.
- If some sessions do not follow the expected order, they may reflect tracking delays, repeated events, checkout behavior, or cases where the first recorded event of a type is not the true first user action.
- These sessions should be documented, not automatically removed, unless there is clear evidence that they are invalid records.


## How missing funnel steps will be handled before EDA

Missing funnel steps will be retained for analysis instead of being removed or forced into a clean funnel path.

Some sessions contain add-to-cart without a same-session view, or purchase without a same-session add-to-cart. These cases may represent valid user behavior, such as:

- viewing a product in one session and adding it to cart in a later session
- adding a product to cart in one session and purchasing it in a later session
- returning directly to cart or checkout
- using a saved cart or checkout flow
- activity that happened before the dataset observation window

They may also reflect tracking limitations, such as missing view or add-to-cart events.

For this reason, these sessions should be documented as funnel interpretation limitations rather than treated as invalid records. During EDA, the analysis can use both:

- a session-level behavioral funnel, which keeps all sessions as observed
- a stricter funnel view, which only counts sessions that contain the expected prior funnel steps

This keeps the analysis honest while preserving potentially valid purchase behavior.


# 5. Exploratory Data Analysis

Start EDA with a simple behavioral segmentation of sessions. This keeps the first EDA step focused on understanding what types of sessions are present before calculating deeper conversion metrics.


## EDA 1: Session behavior type distribution

In [ ]:
import numpy as np

# Create readable session behavior labels from the funnel flags
conditions = [
    (funnel_df['viewed'] == 1) & (funnel_df['carted'] == 0) & (funnel_df['purchased'] == 0),
    (funnel_df['viewed'] == 1) & (funnel_df['carted'] == 1) & (funnel_df['purchased'] == 0),
    (funnel_df['viewed'] == 1) & (funnel_df['carted'] == 1) & (funnel_df['purchased'] == 1),
    (funnel_df['viewed'] == 0) & (funnel_df['carted'] == 1) & (funnel_df['purchased'] == 0),
    (funnel_df['viewed'] == 1) & (funnel_df['carted'] == 0) & (funnel_df['purchased'] == 1),
    (funnel_df['viewed'] == 0) & (funnel_df['carted'] == 1) & (funnel_df['purchased'] == 1),
    (funnel_df['viewed'] == 0) & (funnel_df['carted'] == 0) & (funnel_df['purchased'] == 1)
]

choices = [
    'view_only',
    'view_cart_no_purchase',
    'full_funnel',
    'cart_without_view',
    'view_purchase_no_cart',
    'cart_purchase_no_view',
    'purchase_only'
]

funnel_df['session_type'] = np.select(conditions, choices, default='other')

session_type_summary = (
    funnel_df['session_type']
    .value_counts()
    .rename_axis('session_type')
    .reset_index(name='session_count')
)

session_type_summary['share_of_sessions'] = (
    session_type_summary['session_count'] / funnel_df.shape[0]
).map(lambda x: f'{x:.2%}')

session_type_summary


Interpretation guide:

- `view_only` sessions represent browsing without deeper funnel progression.
- `view_cart_no_purchase` sessions represent cart abandonment opportunities.
- `full_funnel` sessions contain all three key funnel actions in the same session.
- `cart_without_view`, `view_purchase_no_cart`, `cart_purchase_no_view`, and `purchase_only` sessions should be retained but interpreted with the validation caveats documented above.

This table gives the first high-level view of user behavior before moving into conversion rates, drop-offs, and time-based patterns.


The majority of sessions stop at product viewing. Only a small share progresses to cart, and an even smaller share completes purchase in the same session. This suggests the largest funnel opportunity is improving view-to-cart conversion, followed by reducing cart abandonment.

In [ ]:
# Visualize session behavior distribution
import matplotlib.pyplot as plt

plot_df = session_type_summary.copy()
plot_df['share_numeric'] = plot_df['session_count'] / funnel_df.shape[0]
plot_df = plot_df.sort_values('share_numeric', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df['session_type'],
    plot_df['share_numeric'],
    color='#4C78A8'
)

ax.set_title('Session Behavior Type Distribution')
ax.set_xlabel('Share of sessions')
ax.set_ylabel('Session type')
ax.xaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')

for bar, value, count in zip(bars, plot_df['share_numeric'], plot_df['session_count']):
    ax.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f' {value:.2%} ({count:,.0f})',
        va='center',
        fontsize=9
    )

plt.tight_layout()
plt.show()


### EDA 1 interpretation

The session behavior distribution shows that most sessions are `view_only`, which is expected in ecommerce browsing behavior. Only a small share of sessions progress to add-to-cart, and an even smaller share complete purchase in the same session.

Key takeaway: the largest funnel opportunity appears to be improving view-to-cart conversion, followed by reducing cart abandonment.


### Business problem alignment before next EDA

The ecommerce platform needs to understand how users behave within shopping sessions and identify where session-level drop-offs occur between browsing, cart activity, and purchase completion.

This analysis is not trying to prove that the same product moved through `view -> addtocart -> transaction`. Instead, it answers a session-level business question:

> During a shopping session, how far did the user move in the purchase journey?

Business interpretation:

- If most sessions are `view_only`, the main problem is likely before cart. This may point to product page quality, pricing clarity, product discovery, recommendations, trust signals, or call-to-action effectiveness.
- If many sessions are `view_cart_no_purchase`, the problem is likely cart or checkout friction, such as shipping cost, payment issues, long checkout flow, delivery concerns, or lack of trust.
- If purchase sessions are missing prior funnel steps, this may reflect cross-session behavior, tracking gaps, or direct cart/checkout flows.

Objective:

> Build a session-level behavioral funnel to classify shopping sessions, quantify how many sessions contain view/cart/purchase behavior, identify major drop-off patterns, and generate recommendations to improve conversion.


### The project can then have two layers:

1. Session-level behavioral funnel

* Understands overall shopping-session behavior.
* Good for broad user journey/drop-off analysis.

2. Product-session-level funnel

* Understands same-product progression.
* Better for true product funnel conversion.
* Better for product/category-level recommendations.

## Metric 1: Session-level View-to-Cart Conversion

Definition:

> Among sessions that contained at least one `view` event, what percentage also contained at least one `addtocart` event?

Formula:

`Session-level View-to-Cart Conversion = Sessions with view and cart / Sessions with view`

Important interpretation:

This does not prove that the same product was viewed and added to cart. It means the session contained both browsing behavior and cart activity.


In [ ]:
# Metric 1: Session-level View-to-Cart Conversion
view_sessions = funnel_df[funnel_df['viewed'] == 1].shape[0]
view_and_cart_sessions = funnel_df[
    (funnel_df['viewed'] == 1)
    & (funnel_df['carted'] == 1)
].shape[0]

view_to_cart_conversion = view_and_cart_sessions / view_sessions

view_to_cart_summary = pd.DataFrame({
    'metric': [
        'Sessions with view',
        'Sessions with view and cart',
        'Session-level View-to-Cart Conversion'
    ],
    'value': [
        view_sessions,
        view_and_cart_sessions,
        view_to_cart_conversion
    ]
})

view_to_cart_summary['formatted_value'] = [
    f'{view_sessions:,.0f}',
    f'{view_and_cart_sessions:,.0f}',
    f'{view_to_cart_conversion:.2%}'
]

view_to_cart_summary[['metric', 'formatted_value']]


Interpretation guide:

* A low session-level View-to-Cart conversion means many sessions include browsing behavior but do not include cart activity. This may point to opportunities around product page quality, pricing clarity, product discovery, recommendations, trust signals, or call-to-action effectiveness.


## Metric 2: Session-level Cart-to-Purchase Conversion

Definition:

> Among sessions that contained at least one `addtocart` event, what percentage also contained at least one `transaction` event?

Formula:

`Session-level Cart-to-Purchase Conversion = Sessions with cart and purchase / Sessions with cart`

Important interpretation:

This does not prove that the same product was added to cart and purchased. It means the session contained both cart activity and purchase activity.


In [ ]:
# Metric 2: Session-level Cart-to-Purchase Conversion
cart_sessions = funnel_df[funnel_df['carted'] == 1].shape[0]
cart_and_purchase_sessions = funnel_df[
    (funnel_df['carted'] == 1)
    & (funnel_df['purchased'] == 1)
].shape[0]

cart_to_purchase_conversion = cart_and_purchase_sessions / cart_sessions

cart_to_purchase_summary = pd.DataFrame({
    'metric': [
        'Sessions with cart',
        'Sessions with cart and purchase',
        'Session-level Cart-to-Purchase Conversion'
    ],
    'value': [
        cart_sessions,
        cart_and_purchase_sessions,
        cart_to_purchase_conversion
    ]
})

cart_to_purchase_summary['formatted_value'] = [
    f'{cart_sessions:,.0f}',
    f'{cart_and_purchase_sessions:,.0f}',
    f'{cart_to_purchase_conversion:.2%}'
]

cart_to_purchase_summary[['metric', 'formatted_value']]


Interpretation guide:

A low session-level Cart-to-Purchase conversion means many sessions include cart activity but do not include purchase completion. This may point to cart or checkout friction, such as shipping cost, payment issues, long checkout flow, delivery concerns, account creation, or lack of trust.


## Metric 3: Session-level Funnel Drop-off Percentage

Definition:

> Funnel drop-off percentage shows the share of sessions that do not continue from one session-level funnel stage to the next.

For this session-level analysis, the stages are based on whether a session contains each event type:

- `view`
- `addtocart`
- `transaction`

Formulas:

`View-to-Cart Drop-off = 1 - Session-level View-to-Cart Conversion`

`Cart-to-Purchase Drop-off = 1 - Session-level Cart-to-Purchase Conversion`

Important interpretation:

This measures drop-off between session-level behavior groups. It does not prove same-product progression or ordered event sequence.


In [ ]:
# Metric 3: Session-level Funnel Drop-off Percentage
view_to_cart_dropoff = 1 - view_to_cart_conversion
cart_to_purchase_dropoff = 1 - cart_to_purchase_conversion

dropoff_summary = pd.DataFrame({
    'metric': [
        'View-to-Cart Drop-off',
        'Cart-to-Purchase Drop-off'
    ],
    'dropoff_rate': [
        view_to_cart_dropoff,
        cart_to_purchase_dropoff
    ]
})

dropoff_summary['formatted_dropoff_rate'] = dropoff_summary['dropoff_rate'].map(lambda x: f'{x:.2%}')

dropoff_summary[['metric', 'formatted_dropoff_rate']]


Interpretation guide:

A high View-to-Cart drop-off means many sessions include browsing but no cart activity. A high Cart-to-Purchase drop-off means many cart sessions do not include purchase completion.

Comparing these two drop-off rates helps identify whether the larger business opportunity is before cart or after cart.


## Metric 4: Average Session Duration

Definition:

> Average session duration measures how long user sessions last on average.

Formula:

`Average Session Duration = Total session duration / Number of sessions`

Important interpretation:

Session duration is calculated as `session_end - session_start`. A session can be longer than 30 minutes as long as each consecutive event gap stays within the 30-minute inactivity rule. Since averages can be skewed by very long sessions, median session duration is also included.


In [ ]:
# Metric 4: Average Session Duration
avg_session_duration = funnel_df['session_duration_mins'].mean()
median_session_duration = funnel_df['session_duration_mins'].median()

session_duration_metric = pd.DataFrame({
    'metric': [
        'Average session duration',
        'Median session duration'
    ],
    'duration_minutes': [
        avg_session_duration,
        median_session_duration
    ]
})

session_duration_metric['formatted_duration_minutes'] = (
    session_duration_metric['duration_minutes'].map(lambda x: f'{x:.2f} mins')
)

session_duration_metric[['metric', 'formatted_duration_minutes']]


Interpretation guide:

Average session duration shows the typical amount of time users spend in a session, but it can be influenced by unusually long sessions. Median session duration is useful because it shows the midpoint session length and is less affected by outliers.

If purchase or cart sessions have longer durations later, that may suggest users spend more time when they are more engaged.


### Supporting check: single-event sessions

The median session duration is `0.00 mins`, which usually happens when many sessions contain only one event. For a one-event session, `session_start` and `session_end` are the same timestamp, so the calculated duration is 0 minutes.


In [ ]:
# Check the share of sessions with only one event
session_event_counts = (
    df.groupby('session_key')
    .size()
    .reset_index(name='event_count')
)

single_event_session_share = session_event_counts['event_count'].eq(1).mean()
print(f"Share of sessions with only one event: {single_event_session_share:.2%}")


In [ ]:
single_event_sessions = (
    df.groupby('session_key')
    .size()
    .reset_index(name='event_count')
)

single_event_sessions['event_count'].eq(1).mean()

Interpretation:

About 78.28% of sessions have only one event. This explains why the median session duration is `0.00 mins`.

This is not an error. It suggests that most sessions are very short and contain only one recorded action, which aligns with the earlier finding that most sessions are `view_only`.

Business meaning:

> User engagement is shallow for most sessions. Many users browse briefly and leave without additional cart or purchase activity.


In [ ]:
single_event_sessions.head()

## Metric 5: Time Between Session-Level Funnel Steps

Definition:

> Time between steps measures how long it takes within a session to move from one recorded funnel event type to another.

For this session-level analysis, calculate:

- minutes from first `view` to first `addtocart`
- minutes from first `addtocart` to first `transaction`

Important interpretation:

This is a session-level timing metric. It does not prove that the same product was viewed, added to cart, and purchased. To avoid negative or misleading durations, this metric only uses sessions where the first recorded timestamps follow the expected order: `view -> addtocart -> transaction`.


In [ ]:
# Metric 5: Time Between Session-Level Funnel Steps
# Get the first timestamp for each event type within each session.
session_step_times = (
    df[df['event'].isin(['view', 'addtocart', 'transaction'])]
    .groupby(['session_key', 'event'])['timestamp']
    .min()
    .unstack()
    .reset_index()
)

# View -> Cart timing: sessions that have both view and cart, in the expected timestamp order.
view_to_cart_times = session_step_times[
    session_step_times['view'].notna()
    & session_step_times['addtocart'].notna()
    & (session_step_times['view'] <= session_step_times['addtocart'])
].copy()

view_to_cart_times['view_to_cart_mins'] = (
    view_to_cart_times['addtocart'] - view_to_cart_times['view']
).dt.total_seconds() / 60

# Cart -> Purchase timing: sessions that have both cart and purchase, in the expected timestamp order.
cart_to_purchase_times = session_step_times[
    session_step_times['addtocart'].notna()
    & session_step_times['transaction'].notna()
    & (session_step_times['addtocart'] <= session_step_times['transaction'])
].copy()

cart_to_purchase_times['cart_to_purchase_mins'] = (
    cart_to_purchase_times['transaction'] - cart_to_purchase_times['addtocart']
).dt.total_seconds() / 60

time_between_steps_summary = pd.DataFrame({
    'metric': [
        'Sessions used for View -> Cart timing',
        'Average minutes: View -> Cart',
        'Median minutes: View -> Cart',
        'Sessions used for Cart -> Purchase timing',
        'Average minutes: Cart -> Purchase',
        'Median minutes: Cart -> Purchase'
    ],
    'value': [
        view_to_cart_times.shape[0],
        view_to_cart_times['view_to_cart_mins'].mean(),
        view_to_cart_times['view_to_cart_mins'].median(),
        cart_to_purchase_times.shape[0],
        cart_to_purchase_times['cart_to_purchase_mins'].mean(),
        cart_to_purchase_times['cart_to_purchase_mins'].median()
    ]
})

time_between_steps_summary['formatted_value'] = [
    f"{view_to_cart_times.shape[0]:,.0f}",
    f"{view_to_cart_times['view_to_cart_mins'].mean():.2f} mins",
    f"{view_to_cart_times['view_to_cart_mins'].median():.2f} mins",
    f"{cart_to_purchase_times.shape[0]:,.0f}",
    f"{cart_to_purchase_times['cart_to_purchase_mins'].mean():.2f} mins",
    f"{cart_to_purchase_times['cart_to_purchase_mins'].median():.2f} mins"
]

time_between_steps_summary[['metric', 'formatted_value']]


Interpretation guide:

- `View -> Cart` timing shows how quickly sessions with browsing and cart activity move from first view to first cart event.
- `Cart -> Purchase` timing shows how quickly sessions with cart and purchase activity move from first cart event to first transaction event.
- Median is important because average timing can be skewed by unusually long sessions.
- These timings should be interpreted as session-level behavior, not same-product journey timing.


* The largest conversion problem is still view-to-cart, but for users who do add to cart, the purchase step takes longer and may contain checkout friction.

# Descriptive Analytics

### Funnel Performance

The session-level funnel shows strong browsing activity but weak progression into cart and purchase.

1. View-to-Cart conversion is very low at 2.21%, meaning most sessions with product views do not include cart activity.
2. Cart-to-Purchase conversion is stronger at 27.17%, showing that users who add to cart are much more likely to purchase.
3. View-to-Cart drop-off is the largest at 97.79%, making it the biggest funnel leakage point.
4. Cart-to-Purchase drop-off is 72.83%, indicating there is still checkout/cart abandonment, but it is not as severe as the view-to-cart gap.
5. Median session duration is 0.00 mins because 78.28% of sessions have only one event, suggesting most sessions are shallow browsing sessions.
6. For sessions that do progress, users move from View to Cart faster than from Cart to Purchase:
   * View → Cart median time: 2.09 mins
   * Cart → Purchase median time: 4.41 mins
   
Overall:

Funnel performance is limited mainly by weak movement from browsing to cart. The largest opportunity is improving early-stage engagement and motivating users to add products to cart.

### Stage-Wise Drop-Offs

The largest drop-off happens between View → Cart.

View → Cart Drop-off: 97.79%

Most sessions with product views do not include cart activity.
This is the biggest leakage point in the funnel.
Cart → Purchase Drop-off: 72.83%

A large share of cart sessions still do not include purchase activity.
However, this drop-off is lower than the View → Cart drop-off.
Interpretation:

Users are far more likely to drop off before adding to cart than after adding to cart. This suggests the main friction is in moving users from browsing to purchase intent.

Business implication:

The first priority should be improving product discovery, product detail pages, pricing clarity, recommendations, trust signals, and add-to-cart CTA effectiveness. Checkout improvements still matter, but the bigger opportunity is before cart.

## Descriptive Analytics: Purchase vs Non-Purchase Sessions

Purchase is the key business outcome, so compare sessions with purchase activity against sessions without purchase activity.

This section answers:

> How do purchase sessions differ from non-purchase sessions at a high level?

The comparison is still session-level, not product-level.


In [ ]:
# Add event count per session for purchase vs non-purchase comparison
session_event_counts = (
    df.groupby('session_key')
    .size()
    .reset_index(name='event_count')
)

funnel_df = funnel_df.merge(
    session_event_counts,
    on='session_key',
    how='left'
)

purchase_session_comparison = (
    funnel_df
    .assign(session_group=np.where(funnel_df['purchased'] == 1, 'purchase_session', 'non_purchase_session'))
    .groupby('session_group')
    .agg(
        sessions=('session_key', 'count'),
        avg_session_duration_mins=('session_duration_mins', 'mean'),
        median_session_duration_mins=('session_duration_mins', 'median'),
        avg_events_per_session=('event_count', 'mean'),
        median_events_per_session=('event_count', 'median'),
        view_presence_rate=('viewed', 'mean'),
        cart_presence_rate=('carted', 'mean')
    )
    .reset_index()
)

purchase_session_comparison['share_of_sessions'] = (
    purchase_session_comparison['sessions'] / funnel_df.shape[0]
)

purchase_session_comparison_display = purchase_session_comparison.copy()

for col in ['avg_session_duration_mins', 'median_session_duration_mins', 'avg_events_per_session', 'median_events_per_session']:
    purchase_session_comparison_display[col] = purchase_session_comparison_display[col].map(lambda x: f'{x:.2f}')

for col in ['view_presence_rate', 'cart_presence_rate', 'share_of_sessions']:
    purchase_session_comparison_display[col] = purchase_session_comparison_display[col].map(lambda x: f'{x:.2%}')

purchase_session_comparison_display


### Purchase vs non-purchase interpretation

Purchase sessions are very different from non-purchase sessions.

Key observations:

- Purchase sessions are rare, representing only `0.81%` of all sessions.
- Non-purchase sessions dominate the dataset, representing `99.19%` of all sessions.
- Purchase sessions show much deeper engagement:
  - average session duration is `28.78 mins` compared with `1.53 mins` for non-purchase sessions
  - median session duration is `11.70 mins` compared with `0.00 mins` for non-purchase sessions
  - average events per session is `9.34` compared with `1.50` for non-purchase sessions
  - median events per session is `4.00` compared with `1.00` for non-purchase sessions
- Cart presence is much higher in purchase sessions: `83.46%` versus `1.83%` in non-purchase sessions.
- Purchase sessions have `89.61%` view presence, not 100%, which supports the earlier validation finding that some purchase sessions do not contain a view event in the same session.

Business interpretation:

> Converting sessions show much deeper engagement than non-converting sessions. Users who purchase spend more time, generate more events, and are much more likely to have cart activity. The business opportunity is to move more shallow browsing sessions toward deeper engagement and cart activity.


Interpretation guide:

- Purchase sessions represent the converting sessions, while non-purchase sessions represent sessions that did not include a transaction.
- Compare average and median duration to understand whether purchase sessions involve longer engagement.
- Compare event count to see whether purchase sessions are more active.
- Compare cart presence rate to understand how often purchase sessions also include cart activity.

Expected business read:

> Purchase sessions should generally show deeper engagement than non-purchase sessions, through longer duration, more events, and stronger cart presence. If that pattern appears, it supports the idea that engagement depth is linked with conversion.


In [ ]:
funnel_df.head()

## Descriptive Analytics Visual Summary

Use a few focused visuals to summarize the descriptive metrics. These charts are session-level and should not be interpreted as same-product funnel progression.


### Conversion and drop-off comparison

In [ ]:
# Visualize session-level conversion and drop-off rates
conversion_dropoff_plot = pd.DataFrame({
    'metric': [
        'View-to-Cart Conversion',
        'Cart-to-Purchase Conversion',
        'View-to-Cart Drop-off',
        'Cart-to-Purchase Drop-off'
    ],
    'rate': [
        view_to_cart_conversion,
        cart_to_purchase_conversion,
        view_to_cart_dropoff,
        cart_to_purchase_dropoff
    ],
    'type': ['Conversion', 'Conversion', 'Drop-off', 'Drop-off']
})

fig, ax = plt.subplots(figsize=(9, 5))
colors = conversion_dropoff_plot['type'].map({
    'Conversion': '#54A24B',
    'Drop-off': '#E45756'
})

bars = ax.bar(
    conversion_dropoff_plot['metric'],
    conversion_dropoff_plot['rate'],
    color=colors
)

ax.set_title('Session-Level Conversion and Drop-off Rates')
ax.set_ylabel('Rate')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
plt.xticks(rotation=20, ha='right')

for bar, rate in zip(bars, conversion_dropoff_plot['rate']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{rate:.2%}',
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()


Interpretation:

The chart highlights that View-to-Cart drop-off is the largest leakage point. Cart-to-Purchase conversion is stronger than View-to-Cart conversion, so the biggest opportunity is moving more browsing sessions into cart activity.


### Session duration and event count distributions

These charts show how session duration and event count are distributed across sessions. Because most sessions are short and many sessions contain only one event, the charts use capped x-axis ranges to keep the patterns readable.


In [ ]:
# Distribution of session duration and event count per session
# Ensure event_count exists in funnel_df.
if 'event_count' not in funnel_df.columns:
    session_event_counts = (
        df.groupby('session_key')
        .size()
        .reset_index(name='event_count')
    )

    funnel_df = funnel_df.merge(
        session_event_counts,
        on='session_key',
        how='left'
    )

# Cap x-axis at the 99th percentile so extreme long sessions do not flatten the chart.
duration_cap = funnel_df['session_duration_mins'].quantile(0.99)
event_count_cap = funnel_df['event_count'].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(
    data=funnel_df[funnel_df['session_duration_mins'] <= duration_cap],
    x='session_duration_mins',
    bins=40,
    color='#4C78A8',
    ax=axes[0]
)
axes[0].set_title('Session Duration Distribution')
axes[0].set_xlabel('Session duration (minutes)')
axes[0].set_ylabel('Number of sessions')
axes[0].axvline(
    funnel_df['session_duration_mins'].median(),
    color='#E45756',
    linestyle='--',
    label=f"Median: {funnel_df['session_duration_mins'].median():.2f} mins"
)
axes[0].legend()

sns.histplot(
    data=funnel_df[funnel_df['event_count'] <= event_count_cap],
    x='event_count',
    bins=range(1, int(event_count_cap) + 2),
    color='#F58518',
    ax=axes[1]
)
axes[1].set_title('Event Count per Session Distribution')
axes[1].set_xlabel('Events per session')
axes[1].set_ylabel('Number of sessions')
axes[1].axvline(
    funnel_df['event_count'].median(),
    color='#E45756',
    linestyle='--',
    label=f"Median: {funnel_df['event_count'].median():.0f} event"
)
axes[1].legend()

plt.tight_layout()
plt.show()


Interpretation:

The distributions should show that most sessions are short and contain very few events. This supports the descriptive finding that user engagement is shallow for most sessions, with many users leaving after only one recorded action.


### Median time between session-level funnel steps

In [ ]:
# Visualize median time between session-level funnel steps
time_between_steps_plot = pd.DataFrame({
    'step': ['View -> Cart', 'Cart -> Purchase'],
    'median_minutes': [
        view_to_cart_times['view_to_cart_mins'].median(),
        cart_to_purchase_times['cart_to_purchase_mins'].median()
    ]
})

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    time_between_steps_plot['step'],
    time_between_steps_plot['median_minutes'],
    color=['#4C78A8', '#F58518']
)

ax.set_title('Median Time Between Session-Level Funnel Steps')
ax.set_ylabel('Median minutes')
ax.set_xlabel('Step')

for bar, value in zip(bars, time_between_steps_plot['median_minutes']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f'{value:.2f} mins',
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()


Interpretation:

The median Cart-to-Purchase time is longer than the median View-to-Cart time. Users who add to cart tend to do so quickly, while purchase completion takes longer and may involve checkout review, payment decisions, shipping evaluation, or final hesitation.

### Purchase vs non-purchase engagement comparison

In [ ]:
# Visualize engagement differences between purchase and non-purchase sessions
engagement_plot = purchase_session_comparison.copy()
engagement_plot['session_group'] = engagement_plot['session_group'].replace({
    'non_purchase_session': 'Non-purchase',
    'purchase_session': 'Purchase'
})

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

charts = [
    ('avg_session_duration_mins', 'Avg Session Duration (mins)', '#4C78A8'),
    ('avg_events_per_session', 'Avg Events per Session', '#F58518'),
    ('cart_presence_rate', 'Cart Presence Rate', '#54A24B')
]

for ax, (column, title, color) in zip(axes, charts):
    bars = ax.bar(
        engagement_plot['session_group'],
        engagement_plot[column],
        color=color
    )
    ax.set_title(title)
    ax.set_xlabel('Session group')
    ax.set_ylabel('Value')

    if column == 'cart_presence_rate':
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
        labels = [f'{value:.2%}' for value in engagement_plot[column]]
    else:
        labels = [f'{value:.2f}' for value in engagement_plot[column]]

    for bar, label in zip(bars, labels):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )

plt.tight_layout()
plt.show()


Interpretation:

Purchase sessions show deeper engagement than non-purchase sessions. They last longer, contain more events, and have much higher cart presence. This supports the idea that increasing engagement depth and cart activity may improve conversion.

# 6. Diagnostic Analytics

Descriptive analytics showed that most sessions are shallow, while purchase sessions have longer duration and more events. Diagnostic analytics starts asking why conversion differs across groups.

First question:

> Do more engaged sessions convert better than low-engagement sessions?


## Diagnostic 1: Conversion by event-count segment

In [ ]:
# Ensure event_count exists in funnel_df.
if 'event_count' not in funnel_df.columns:
    session_event_counts = (
        df.groupby('session_key')
        .size()
        .reset_index(name='event_count')
    )

    funnel_df = funnel_df.merge(
        session_event_counts,
        on='session_key',
        how='left'
    )

# Segment sessions by engagement depth using event count.
funnel_df['event_count_segment'] = pd.cut(
    funnel_df['event_count'],
    bins=[0, 1, 3, 10, float('inf')],
    labels=['1 event', '2-3 events', '4-10 events', '11+ events']
)

event_count_segment_summary = (
    funnel_df
    .groupby('event_count_segment', observed=True)
    .agg(
        sessions=('session_key', 'count'),
        view_rate=('viewed', 'mean'),
        cart_rate=('carted', 'mean'),
        purchase_rate=('purchased', 'mean'),
        avg_session_duration_mins=('session_duration_mins', 'mean')
    )
    .reset_index()
)

event_count_segment_summary['share_of_sessions'] = (
    event_count_segment_summary['sessions'] / funnel_df.shape[0]
)

event_count_segment_display = event_count_segment_summary.copy()

for col in ['view_rate', 'cart_rate', 'purchase_rate', 'share_of_sessions']:
    event_count_segment_display[col] = event_count_segment_display[col].map(lambda x: f'{x:.2%}')

event_count_segment_display['avg_session_duration_mins'] = (
    event_count_segment_display['avg_session_duration_mins'].map(lambda x: f'{x:.2f}')
)

event_count_segment_display


In [ ]:
# Visualize cart and purchase rates by event-count segment
plot_df = event_count_segment_summary.melt(
    id_vars='event_count_segment',
    value_vars=['cart_rate', 'purchase_rate'],
    var_name='metric',
    value_name='rate'
)

plot_df['metric'] = plot_df['metric'].replace({
    'cart_rate': 'Cart rate',
    'purchase_rate': 'Purchase rate'
})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=plot_df,
    x='event_count_segment',
    y='rate',
    hue='metric',
    palette=['#F58518', '#54A24B'],
    ax=ax
)

ax.set_title('Cart and Purchase Rates by Event-Count Segment')
ax.set_xlabel('Event-count segment')
ax.set_ylabel('Rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')

for container in ax.containers:
    labels = [f'{bar.get_height():.2%}' for bar in container]
    ax.bar_label(container, labels=labels, padding=3, fontsize=9)

plt.tight_layout()
plt.show()


Interpretation guide:

If cart and purchase rates increase as event count increases, it suggests that engagement depth is strongly associated with conversion. This would support the descriptive finding that purchase sessions are more active than non-purchase sessions.

Business implication:

> The business should focus on moving users from single-event browsing sessions into deeper engagement, because sessions with more actions are more likely to include cart and purchase behavior.


## Diagnostic 2: Conversion by session-duration segment

Descriptive analytics showed that purchase sessions have much longer durations than non-purchase sessions. This diagnostic check compares cart and purchase rates across session-duration segments.

Question:

> Do longer sessions have higher cart and purchase rates?


In [ ]:
# Segment sessions by duration and compare cart/purchase rates
funnel_df['duration_segment'] = pd.cut(
    funnel_df['session_duration_mins'],
    bins=[-0.01, 0, 2, 10, float('inf')],
    labels=['0 mins', '0-2 mins', '2-10 mins', '10+ mins']
)

duration_segment_summary = (
    funnel_df
    .groupby('duration_segment', observed=True)
    .agg(
        sessions=('session_key', 'count'),
        avg_events_per_session=('event_count', 'mean'),
        cart_rate=('carted', 'mean'),
        purchase_rate=('purchased', 'mean')
    )
    .reset_index()
)

duration_segment_summary['share_of_sessions'] = (
    duration_segment_summary['sessions'] / funnel_df.shape[0]
)

duration_segment_display = duration_segment_summary.copy()

duration_segment_display['avg_events_per_session'] = (
    duration_segment_display['avg_events_per_session'].map(lambda x: f'{x:.2f}')
)

for col in ['cart_rate', 'purchase_rate', 'share_of_sessions']:
    duration_segment_display[col] = duration_segment_display[col].map(lambda x: f'{x:.2%}')

duration_segment_display


In [ ]:
# Visualize cart and purchase rates by session-duration segment
plot_df = duration_segment_summary.melt(
    id_vars='duration_segment',
    value_vars=['cart_rate', 'purchase_rate'],
    var_name='metric',
    value_name='rate'
)

plot_df['metric'] = plot_df['metric'].replace({
    'cart_rate': 'Cart rate',
    'purchase_rate': 'Purchase rate'
})

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=plot_df,
    x='duration_segment',
    y='rate',
    hue='metric',
    palette=['#F58518', '#54A24B'],
    ax=ax
)

ax.set_title('Cart and Purchase Rates by Session-Duration Segment')
ax.set_xlabel('Session-duration segment')
ax.set_ylabel('Rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')

for container in ax.containers:
    labels = [f'{bar.get_height():.2%}' for bar in container]
    ax.bar_label(container, labels=labels, padding=3, fontsize=9)

plt.tight_layout()
plt.show()


Interpretation guide:

If cart and purchase rates increase for longer-duration sessions, it suggests that deeper session engagement is associated with conversion. This does not prove causality, but it helps identify session engagement as an important conversion signal.


## Diagnostic 3: View-to-Cart Drop-off Analysis

View-to-Cart drop-off is the largest funnel leakage point, so inspect viewed sessions more closely.

Question:

> How do sessions with view-only behavior differ from sessions that include both view and cart activity?


In [ ]:
# Compare viewed sessions with and without cart activity
view_sessions_df = funnel_df[funnel_df['viewed'] == 1].copy()

view_sessions_df['view_session_group'] = np.where(
    view_sessions_df['carted'] == 1,
    'view_with_cart',
    'view_only_no_cart'
)

view_to_cart_dropoff_summary = (
    view_sessions_df
    .groupby('view_session_group')
    .agg(
        sessions=('session_key', 'count'),
        avg_session_duration_mins=('session_duration_mins', 'mean'),
        median_session_duration_mins=('session_duration_mins', 'median'),
        avg_events_per_session=('event_count', 'mean'),
        median_events_per_session=('event_count', 'median'),
        purchase_presence_rate=('purchased', 'mean')
    )
    .reset_index()
)

view_to_cart_dropoff_summary['share_of_view_sessions'] = (
    view_to_cart_dropoff_summary['sessions'] / view_sessions_df.shape[0]
)

view_to_cart_dropoff_display = view_to_cart_dropoff_summary.copy()

for col in ['avg_session_duration_mins', 'median_session_duration_mins', 'avg_events_per_session', 'median_events_per_session']:
    view_to_cart_dropoff_display[col] = view_to_cart_dropoff_display[col].map(lambda x: f'{x:.2f}')

for col in ['purchase_presence_rate', 'share_of_view_sessions']:
    view_to_cart_dropoff_display[col] = view_to_cart_dropoff_display[col].map(lambda x: f'{x:.2%}')

view_to_cart_dropoff_display


In [ ]:
# Visualize view-to-cart drop-off group comparison
plot_df = view_to_cart_dropoff_summary.copy()
plot_df['view_session_group'] = plot_df['view_session_group'].replace({
    'view_only_no_cart': 'View only / no cart',
    'view_with_cart': 'View with cart'
})

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

charts = [
    ('sessions', 'Viewed Session Count', '#4C78A8'),
    ('avg_session_duration_mins', 'Avg Session Duration (mins)', '#F58518'),
    ('avg_events_per_session', 'Avg Events per Session', '#54A24B')
]

for ax, (column, title, color) in zip(axes, charts):
    bars = ax.bar(
        plot_df['view_session_group'],
        plot_df[column],
        color=color
    )
    ax.set_title(title)
    ax.set_xlabel('Viewed session group')
    ax.set_ylabel('Value')

    labels = [f'{value:,.0f}' if column == 'sessions' else f'{value:.2f}' for value in plot_df[column]]
    for bar, label in zip(bars, labels):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )

plt.tight_layout()
plt.show()


Interpretation guide:

Compare `view_only_no_cart` sessions with `view_with_cart` sessions to understand the largest drop-off point.

If `view_with_cart` sessions have longer duration or more events, it suggests that deeper browsing engagement is associated with cart activity. If most viewed sessions remain `view_only_no_cart`, the business opportunity is to move users from passive browsing into stronger purchase intent.


## Diagnostic 4: Cart abandonment analysis

Cart-to-Purchase drop-off is high, so inspect sessions with cart activity more closely.

Question:

> How do cart sessions that end with purchase differ from cart sessions that do not include purchase?


In [ ]:
# Compare cart sessions with and without purchase
cart_sessions_df = funnel_df[funnel_df['carted'] == 1].copy()

cart_sessions_df['cart_session_group'] = np.where(
    cart_sessions_df['purchased'] == 1,
    'cart_with_purchase',
    'cart_no_purchase'
)

cart_abandonment_summary = (
    cart_sessions_df
    .groupby('cart_session_group')
    .agg(
        sessions=('session_key', 'count'),
        avg_session_duration_mins=('session_duration_mins', 'mean'),
        median_session_duration_mins=('session_duration_mins', 'median'),
        avg_events_per_session=('event_count', 'mean'),
        median_events_per_session=('event_count', 'median'),
        view_presence_rate=('viewed', 'mean')
    )
    .reset_index()
)

cart_abandonment_summary['share_of_cart_sessions'] = (
    cart_abandonment_summary['sessions'] / cart_sessions_df.shape[0]
)

cart_abandonment_display = cart_abandonment_summary.copy()

for col in ['avg_session_duration_mins', 'median_session_duration_mins', 'avg_events_per_session', 'median_events_per_session']:
    cart_abandonment_display[col] = cart_abandonment_display[col].map(lambda x: f'{x:.2f}')

for col in ['view_presence_rate', 'share_of_cart_sessions']:
    cart_abandonment_display[col] = cart_abandonment_display[col].map(lambda x: f'{x:.2%}')

cart_abandonment_display


In [ ]:
# Visualize cart abandonment group comparison
plot_df = cart_abandonment_summary.copy()
plot_df['cart_session_group'] = plot_df['cart_session_group'].replace({
    'cart_no_purchase': 'Cart no purchase',
    'cart_with_purchase': 'Cart with purchase'
})

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

charts = [
    ('sessions', 'Cart Session Count', '#4C78A8'),
    ('avg_session_duration_mins', 'Avg Session Duration (mins)', '#F58518'),
    ('avg_events_per_session', 'Avg Events per Session', '#54A24B')
]

for ax, (column, title, color) in zip(axes, charts):
    bars = ax.bar(
        plot_df['cart_session_group'],
        plot_df[column],
        color=color
    )
    ax.set_title(title)
    ax.set_xlabel('Cart session group')
    ax.set_ylabel('Value')

    labels = [f'{value:,.0f}' if column == 'sessions' else f'{value:.2f}' for value in plot_df[column]]
    for bar, label in zip(bars, labels):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )

plt.tight_layout()
plt.show()


Interpretation guide:

Compare cart sessions with and without purchase to understand checkout-stage friction. If cart-with-purchase sessions are longer or have more events, it may suggest that deeper engagement supports purchase completion. If cart-no-purchase sessions dominate, checkout or cart-stage friction remains a meaningful business opportunity.


Most cart sessions do not end in purchase, but cart sessions that do convert show significantly deeper engagement. This suggests that increasing engagement after cart activity and reducing checkout friction may help improve purchase completion.

## Diagnostic 5: Conversion by visitor frequency

Some users may browse in one session and convert in a later session. This diagnostic check compares one-session visitors with repeat visitors.

Question:

> Do repeat visitors show stronger cart and purchase behavior than one-session visitors?


In [ ]:
# Compare conversion behavior for one-session visitors vs repeat visitors
visitor_session_counts = (
    funnel_df
    .groupby('visitorid')['session_key']
    .nunique()
    .reset_index(name='session_count_per_visitor')
)

visitor_outcomes = (
    funnel_df
    .groupby('visitorid')
    .agg(
        visitor_sessions=('session_key', 'count'),
        had_view=('viewed', 'max'),
        had_cart=('carted', 'max'),
        had_purchase=('purchased', 'max'),
        avg_session_duration_mins=('session_duration_mins', 'mean'),
        avg_events_per_session=('event_count', 'mean')
    )
    .reset_index()
)

visitor_outcomes = visitor_outcomes.merge(
    visitor_session_counts,
    on='visitorid',
    how='left'
)

visitor_outcomes['visitor_frequency_segment'] = np.where(
    visitor_outcomes['session_count_per_visitor'] == 1,
    'one_session_visitor',
    'repeat_visitor'
)

visitor_frequency_summary = (
    visitor_outcomes
    .groupby('visitor_frequency_segment')
    .agg(
        visitors=('visitorid', 'count'),
        avg_sessions_per_visitor=('session_count_per_visitor', 'mean'),
        view_rate=('had_view', 'mean'),
        cart_rate=('had_cart', 'mean'),
        purchase_rate=('had_purchase', 'mean'),
        avg_session_duration_mins=('avg_session_duration_mins', 'mean'),
        avg_events_per_session=('avg_events_per_session', 'mean')
    )
    .reset_index()
)

visitor_frequency_summary['share_of_visitors'] = (
    visitor_frequency_summary['visitors'] / visitor_outcomes.shape[0]
)

visitor_frequency_display = visitor_frequency_summary.copy()

for col in ['avg_sessions_per_visitor', 'avg_session_duration_mins', 'avg_events_per_session']:
    visitor_frequency_display[col] = visitor_frequency_display[col].map(lambda x: f'{x:.2f}')

for col in ['view_rate', 'cart_rate', 'purchase_rate', 'share_of_visitors']:
    visitor_frequency_display[col] = visitor_frequency_display[col].map(lambda x: f'{x:.2%}')

visitor_frequency_display


In [ ]:
# Visualize cart and purchase rates by visitor frequency segment
plot_df = visitor_frequency_summary.melt(
    id_vars='visitor_frequency_segment',
    value_vars=['cart_rate', 'purchase_rate'],
    var_name='metric',
    value_name='rate'
)

plot_df['visitor_frequency_segment'] = plot_df['visitor_frequency_segment'].replace({
    'one_session_visitor': 'One-session visitor',
    'repeat_visitor': 'Repeat visitor'
})

plot_df['metric'] = plot_df['metric'].replace({
    'cart_rate': 'Cart rate',
    'purchase_rate': 'Purchase rate'
})

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=plot_df,
    x='visitor_frequency_segment',
    y='rate',
    hue='metric',
    palette=['#F58518', '#54A24B'],
    ax=ax
)

ax.set_title('Cart and Purchase Rates by Visitor Frequency')
ax.set_xlabel('Visitor frequency segment')
ax.set_ylabel('Rate')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')

for container in ax.containers:
    labels = [f'{bar.get_height():.2%}' for bar in container]
    ax.bar_label(container, labels=labels, padding=3, fontsize=9)

plt.tight_layout()
plt.show()


Interpretation guide:

If repeat visitors have higher cart or purchase rates, it suggests that returning behavior is associated with stronger purchase intent. This would support the idea that some ecommerce conversion happens across multiple sessions rather than in a single session.

Business implication:

> Encouraging users to return through remarketing, saved carts, personalized recommendations, email reminders, or improved account experiences may help move users from browsing toward purchase.


# 7. Inferential Statistics

Descriptive and diagnostic analytics suggested that deeper session engagement is associated with conversion. Inferential statistics can help test whether the observed differences are statistically significant, not just descriptive patterns.


## Test 1: Two-proportion z-test for purchase rate by engagement level

Diagnostic analytics showed that sessions with more events appear more likely to include purchase activity.

Question:

> Do multi-event sessions have a significantly higher purchase rate than single-event sessions?

Groups:

- Single-event sessions: sessions with exactly 1 event
- Multi-event sessions: sessions with more than 1 event

Outcome:

- `purchased = 1` if the session contains a transaction event
- `purchased = 0` otherwise

Hypotheses:

- Null hypothesis: Multi-event sessions and single-event sessions have the same purchase rate.
- Alternative hypothesis: Multi-event sessions have a higher purchase rate than single-event sessions.


In [ ]:
# Test 1: Two-proportion z-test for purchase rate by engagement level
from statsmodels.stats.proportion import proportions_ztest

# Ensure event_count exists in funnel_df.
if 'event_count' not in funnel_df.columns:
    session_event_counts = (
        df.groupby('session_key')
        .size()
        .reset_index(name='event_count')
    )

    funnel_df = funnel_df.merge(
        session_event_counts,
        on='session_key',
        how='left'
    )

single_event_sessions = funnel_df[funnel_df['event_count'] == 1]
multi_event_sessions = funnel_df[funnel_df['event_count'] > 1]

single_event_purchases = single_event_sessions['purchased'].sum()
multi_event_purchases = multi_event_sessions['purchased'].sum()

single_event_total = single_event_sessions.shape[0]
multi_event_total = multi_event_sessions.shape[0]

single_event_purchase_rate = single_event_purchases / single_event_total
multi_event_purchase_rate = multi_event_purchases / multi_event_total

# Two-proportion z-test
# H1: multi-event sessions have a higher purchase rate than single-event sessions.
count = [multi_event_purchases, single_event_purchases]
nobs = [multi_event_total, single_event_total]

z_score, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative='larger'
)

engagement_purchase_test_summary = pd.DataFrame({
    'group': ['Single-event sessions', 'Multi-event sessions'],
    'sessions': [single_event_total, multi_event_total],
    'purchase_sessions': [single_event_purchases, multi_event_purchases],
    'purchase_rate': [single_event_purchase_rate, multi_event_purchase_rate]
})

engagement_purchase_test_summary['purchase_rate'] = (
    engagement_purchase_test_summary['purchase_rate'].map(lambda x: f'{x:.4%}')
)

print(f"Z-score: {z_score:.4f}")
print(f"One-tailed p-value: {p_value:.6g}")

engagement_purchase_test_summary


Interpretation guide:

If the p-value is below `0.05`, reject the null hypothesis and conclude that multi-event sessions have a statistically higher purchase rate than single-event sessions.

Business interpretation:

> If significant, this supports the diagnostic finding that deeper session engagement is associated with higher conversion. The business should focus on moving users beyond single-event browsing sessions into richer product exploration and cart activity.

Important caveat:

This test shows association, not causation. It does not prove that adding more events causes purchase, only that multi-event sessions are statistically associated with higher purchase rates.


## Test 2: Mann-Whitney U test for session duration by purchase status

Descriptive analytics showed that purchase sessions have much longer session durations than non-purchase sessions.

Question:

> Are purchase sessions significantly longer than non-purchase sessions?

Why Mann-Whitney U test?

Session duration is highly skewed, with many sessions having `0.00` minutes. Because of this, a standard two-sample t-test is not ideal. The Mann-Whitney U test is a non-parametric test that compares the distributions of two independent groups without assuming normality.

Groups:

- Purchase sessions: sessions where `purchased = 1`
- Non-purchase sessions: sessions where `purchased = 0`

Hypotheses:

- Null hypothesis: Purchase and non-purchase sessions have the same session-duration distribution.
- Alternative hypothesis: Purchase sessions have longer session durations than non-purchase sessions.


In [ ]:
# Test 2: Mann-Whitney U test for session duration by purchase status
from scipy.stats import mannwhitneyu

purchase_durations = funnel_df.loc[
    funnel_df['purchased'] == 1,
    'session_duration_mins'
]

non_purchase_durations = funnel_df.loc[
    funnel_df['purchased'] == 0,
    'session_duration_mins'
]

u_statistic, p_value = mannwhitneyu(
    purchase_durations,
    non_purchase_durations,
    alternative='greater'
)

duration_test_summary = pd.DataFrame({
    'session_group': ['Non-purchase sessions', 'Purchase sessions'],
    'sessions': [non_purchase_durations.shape[0], purchase_durations.shape[0]],
    'avg_duration_mins': [non_purchase_durations.mean(), purchase_durations.mean()],
    'median_duration_mins': [non_purchase_durations.median(), purchase_durations.median()]
})

duration_test_display = duration_test_summary.copy()

for col in ['avg_duration_mins', 'median_duration_mins']:
    duration_test_display[col] = duration_test_display[col].map(lambda x: f'{x:.2f} mins')

print(f"U-statistic: {u_statistic:,.0f}")
print(f"One-tailed p-value: {p_value:.6g}")

duration_test_display


Interpretation guide:

If the p-value is below `0.05`, reject the null hypothesis and conclude that purchase sessions have significantly longer session durations than non-purchase sessions.

Business interpretation:

> If significant, this supports the diagnostic finding that deeper engagement is associated with conversion. Purchase sessions are not only descriptively longer; their duration distribution is statistically higher than non-purchase sessions.

Important caveat:

This test shows association, not causation. Longer sessions are associated with purchase behavior, but this does not prove that increasing session duration alone causes users to purchase.


# 8. Key Findings

## 1. Most sessions are shallow browsing sessions

Most sessions are `view_only`, and `78.28%` of sessions contain only one event. This explains why the median session duration is `0.00 mins`.

Key takeaway:

> A large share of users browse briefly and leave without deeper engagement.

## 2. View-to-Cart is the largest funnel leakage point

Session-level View-to-Cart conversion is only `2.21%`, while View-to-Cart drop-off is `97.79%`.

Key takeaway:

> The biggest conversion opportunity is moving users from product viewing into cart activity.

## 3. Cart-to-Purchase conversion is stronger than View-to-Cart conversion

Session-level Cart-to-Purchase conversion is `27.17%`, while Cart-to-Purchase drop-off is `72.83%`.

Key takeaway:

> Once users add items to cart, they are much more likely to purchase than viewed sessions are to add to cart. Cart abandonment still matters, but the bigger issue is before cart.

## 4. Purchase sessions show much deeper engagement

Purchase sessions represent only `0.81%` of sessions, but they have much stronger engagement than non-purchase sessions:

- average session duration: `28.78 mins` vs `1.53 mins`
- median session duration: `11.70 mins` vs `0.00 mins`
- average events per session: `9.34` vs `1.50`
- cart presence rate: `83.46%` vs `1.83%`

Key takeaway:

> Converting sessions are longer, more active, and much more likely to include cart behavior.

## 5. Engagement depth is statistically associated with purchase

The two-proportion z-test showed that multi-event sessions have a significantly higher purchase rate than single-event sessions:

- single-event purchase rate: `0.0388%`
- multi-event purchase rate: `3.5959%`
- p-value: `< 0.001`

Key takeaway:

> Multi-event sessions are statistically more likely to include purchase behavior.

## 6. Purchase sessions are statistically longer than non-purchase sessions

The Mann-Whitney U test showed that purchase sessions have a significantly higher session-duration distribution than non-purchase sessions.

Key takeaway:

> Longer session engagement is strongly associated with purchase behavior.

## 7. This is a session-level behavioral funnel, not a same-product funnel

The analysis measures whether a session contains view, cart, and purchase activity. It does not prove that the same product moved through `view -> addtocart -> transaction`.

Key takeaway:

> The findings are valid for understanding session-level shopping behavior, but product-level funnel analysis should be a future extension.


# 9. Business Recommendations

## 1. Prioritize improving View-to-Cart conversion

Because View-to-Cart drop-off is the largest leakage point, the first priority should be helping more browsing sessions move into cart activity.

Recommended actions:

- Improve product detail pages with clearer pricing, availability, delivery information, and return policies.
- Make add-to-cart CTAs more visible and easier to use.
- Add trust signals such as reviews, ratings, guarantees, and secure checkout messaging.
- Improve product recommendations to help users find relevant items faster.
- Highlight promotions, bundles, or urgency cues where appropriate.

## 2. Increase engagement depth for shallow sessions

Most sessions are single-event sessions, and multi-event sessions are significantly more likely to purchase.

Recommended actions:

- Encourage users to continue browsing with related products, recently viewed items, and personalized recommendations.
- Improve search and category navigation so users can explore more products.
- Use onsite prompts or recommendations to reduce one-and-done browsing sessions.
- Track whether users who interact with recommendations show higher cart or purchase rates.

## 3. Reduce cart and checkout friction

Cart-to-Purchase conversion is stronger than View-to-Cart conversion, but `72.83%` of cart sessions still do not include purchase.

Recommended actions:

- Review checkout steps for unnecessary friction.
- Make shipping costs and delivery dates visible earlier.
- Simplify payment and account creation flows.
- Add reassurance around returns, security, and customer support.
- Consider cart reminders or saved-cart flows for users who leave after cart activity.

## 4. Support returning-user and cross-session behavior

Some users may browse, cart, and purchase across different sessions. Repeat behavior should be supported instead of assuming all conversion happens in one visit.

Recommended actions:

- Use saved carts and persistent cart experiences.
- Add email or notification reminders for cart abandoners.
- Personalize recommendations based on previous browsing and cart behavior.
- Retarget users who showed product interest but did not cart or purchase.

## 5. Improve tracking and future product-level analysis

Some sessions contain purchase activity without same-session view or cart activity. This may reflect cross-session behavior, direct checkout flows, or tracking gaps.

Recommended actions:

- Validate whether view, add-to-cart, and transaction events are consistently tracked.
- Add product-session-level funnel analysis using `visitorid + session_id + itemid`.
- Analyze category-level and product-level conversion once the product-level funnel is created.
- Add device, traffic source, and channel data if available to identify where funnel performance differs.

## Overall recommendation

> The largest opportunity is to improve early-stage engagement and View-to-Cart conversion. Users who become more engaged are much more likely to purchase, so the business should focus on turning shallow browsing sessions into deeper product exploration and cart activity, while also reducing checkout friction for users who reach the cart stage.


## Save Processed Funnel Dataset

Save the validated session-level funnel table so it can be reused by the Tableau Data Preparation notebook and the Logistic Regression workflow.

In [ ]:
# Save validated session-level funnel dataset
from pathlib import Path

processed_output_path = Path('../Dataset/Processed Dataset/funnel_data.csv')
processed_output_path.parent.mkdir(parents=True, exist_ok=True)

funnel_df.to_csv(processed_output_path, index=False)

print(f'Processed funnel dataset saved to: {processed_output_path}')
print(f'Rows saved: {len(funnel_df):,}')
print(f'Columns saved: {len(funnel_df.columns):,}')
print('Columns:')
print(list(funnel_df.columns))

In [ ]:
# Confirm saved file can be read back
saved_funnel_check = pd.read_csv(processed_output_path, nrows=5)
saved_funnel_check.head()